# Compilador `bell` - Trabalho do Grau B (UNISINOS)
**Analisador Léxico (Flex), Sintático (Bison), Verificação Semântica (Tipagem Estrita) e Tradução Dirigida por Sintaxe (Notação Pós-Fixada / RPN)**

Este caderno executa **do zero** no Google Colab sem necessidade de arquivos externos:
1. Instala `flex`, `bison` e `gcc` no ambiente do Colab;
2. Gera os arquivos-fonte (`semantico.h`, `semantico.c`, `analisador.l`, `analisador.y` e `Makefile`);
3. Compila o executável `./analisador` (verificando **0 conflitos** no Bison e **0 warnings** no GCC);
4. Cria e executa os **4 programas de teste válidos** e os **4 programas de teste inválidos** (semânticos, sintáticos e léxicos) na ordem exigida para a demonstração em vídeo.


## 1. Instalação das Ferramentas (Flex, Bison e GCC)


In [ ]:
!apt-get update -qq && apt-get install -y -qq flex bison gcc make
!mkdir -p examples
!bison --version | head -n 1 && flex --version


## 2. Módulo Semântico e Tradução Dirigida (`semantico.h` e `semantico.c`)
Define a **Tabela de Símbolos**, os tipos (`TIPO_INT`, `TIPO_FLOAT`, `TIPO_ERRO`), as funções de verificação semântica e a síntese da **Notação Pós-Fixada (RPN)**.


In [ ]:
%%writefile semantico.h
/* ==========================================================================
   MODULO SEMANTICO E TRADUCAO DIRIGIDA (CABECALHO) -- Linguagem bell
   Trabalho do Grau B -- Compiladores -- UNISINOS

   Por que separar o analisador semantico em um arquivo proprio?
   1) Mantem a gramatica do Bison (analisador.y) limpa e legivel;
   2) Facilita explicar na apresentacao onde ficam a Tabela de Simbolos,
      as regras de verificacao de tipos e a geracao da Notacao Pos-Fixada.
   ========================================================================== */

#ifndef SEMANTICO_H
#define SEMANTICO_H

/* Nome da linguagem projetada pelo grupo (aparece nas mensagens e relatorios). */
#define NOME_LINGUAGEM "bell"

/* Limites das tabelas em memoria. */
#define MAX_SIMBOLOS  200
#define MAX_TRADUCOES 500

/* --------------------------------------------------------------------------
   TIPOS DE DADOS DA LINGUAGEM
   Alem de TIPO_INT e TIPO_FLOAT, criamos o TIPO_ERRO.
   Por que TIPO_ERRO existe?
   Imagine a expressao: (10 + 2.5) + 3
   - Quando avaliamos (10 + 2.5), detectamos erro (int + float) e devolvemos
     TIPO_ERRO como resultado dessa subexpressao.
   - Assim, na soma externa (TIPO_ERRO + 3), sabemos que o erro ja foi avisado
     e evitamos imprimir varios erros repetidos na mesma linha!
   -------------------------------------------------------------------------- */
typedef enum {
    TIPO_INT,
    TIPO_FLOAT,
    TIPO_ERRO
} TipoDado;

/* --------------------------------------------------------------------------
   ATRIBUTO SINTETIZADO PELAS EXPRESSOES (Semantica + Traducao Dirigida)
   Cada no de 'expressao' na arvore do Bison carrega duas informacoes:
   1) tipo : o TipoDado resultante (para a verificacao semantica);
   2) rpn  : o texto em Notacao Pos-Fixada (Polonesa Reversa), ex.: "10 5 2 * +".
   -------------------------------------------------------------------------- */
typedef struct {
    TipoDado tipo;
    char *rpn;
} AtributoExpressao;

/* --------------------------------------------------------------------------
   ESTRUTURA DA TABELA DE SIMBOLOS
   Guarda cada variavel declarada no programa: seu nome, seu tipo e a linha
   onde ela foi declarada.
   -------------------------------------------------------------------------- */
typedef struct {
    char *nome;
    TipoDado tipo;
    int linha_decl;
} Simbolo;

/* Contador global de erros semanticos encontrados durante a analise. */
extern int erros_semanticos;

/* --- Operacoes da Tabela de Simbolos --- */

/* Procura uma variavel pelo nome. Retorna o indice (0..n-1) ou -1 se nao achar. */
int buscar_simbolo(const char *nome);

/* Registra uma nova variavel na Tabela de Simbolos (verifica duplicidade). */
void declarar_variavel(const char *nome, TipoDado tipo, int linha);

/* Consulta o tipo de uma variavel usada em expressao (verifica se foi declarada). */
TipoDado obter_tipo_variavel(const char *nome, int linha);

/* --- Regras de Verificacao Semantica de Tipos --- */

/* Verifica se dois operandos (esq e dir) tem tipos compativeis para +, -, *, /. */
TipoDado verificar_operacao_aritmetica(TipoDado esq, TipoDado dir, const char *op, int linha);

/* Verifica se a variavel do lado esquerdo de '=' existe e tem o mesmo tipo da expressao. */
void verificar_atribuicao(const char *nome_var, TipoDado tipo_expr, int linha);

/* Verifica se os dois lados de uma comparacao (==, !=, >, <, >=, <=) tem o mesmo tipo. */
void verificar_condicao_relacional(TipoDado esq, TipoDado dir, const char *op, int linha);

/* Verifica se a expressao usada em 'switch (expr)' eh do tipo inteiro (int). */
void verificar_selecao_switch(TipoDado tipo_expr, int linha);

/* --- Traducao Dirigida por Sintaxe (Notacao Pos-Fixada / RPN) --- */

/* Junta duas subexpressoes e um operador no formato pos-fixado: "esq dir op"
   e ja libera a memoria das strings antigas (esq e dir). */
char* combinar_rpn(char *esq, char *dir, const char *op);

/* Registra uma linha traduzida em Notacao Pos-Fixada (atribuicao, if, switch). */
void registrar_traducao(int linha, const char *contexto, const char *rpn);

/* Imprime todas as linhas traduzidas em Notacao Pos-Fixada. */
void imprimir_traducao_posfixada(void);

/* --- Utilitarios --- */

/* Converte o enum TipoDado em texto ("int", "float", "erro") para mensagens. */
const char* nome_tipo(TipoDado tipo);

/* Imprime toda a Tabela de Simbolos formatada no final da execucao. */
void imprimir_tabela_simbolos(void);

/* Libera a memoria alocada na Tabela de Simbolos e na lista de traducoes. */
void liberar_memoria_semantica(void);

#endif


In [ ]:
%%writefile semantico.c
/* ==========================================================================
   MODULO SEMANTICO E TRADUCAO DIRIGIDA (IMPLEMENTACAO) -- Linguagem bell
   Trabalho do Grau B -- Compiladores -- UNISINOS
   ========================================================================== */

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include "semantico.h"

/* Tabela de Simbolos em memoria e contador de simbolos cadastrados. */
static Simbolo tabela[MAX_SIMBOLOS];
static int n_simbolos = 0;

/* Lista de instrucoes traduzidas para Notacao Pos-Fixada (RPN). */
typedef struct {
    int linha;
    char *texto;
} LinhaTraduzida;

static LinhaTraduzida traducoes[MAX_TRADUCOES];
static int n_traducoes = 0;

/* Contador global de erros semanticos. */
int erros_semanticos = 0;

/* --------------------------------------------------------------------------
   Funcao auxiliar interna para padronizar mensagens de erro semantico.
   Atende ao requisito do enunciado: identifica a linguagem, a natureza
   do erro (SEMANTICO) e o numero da linha.
   -------------------------------------------------------------------------- */
static void erro_semantico(int linha, const char *msg) {
    printf("  [%s - ERRO SEMANTICO] linha %d: %s\n", NOME_LINGUAGEM, linha, msg);
    erros_semanticos++;
}

/* Converte o enum TipoDado para string legivel. */
const char* nome_tipo(TipoDado tipo) {
    switch (tipo) {
        case TIPO_INT:   return "int";
        case TIPO_FLOAT: return "float";
        default:         return "erro";
    }
}

/* --------------------------------------------------------------------------
   Identificador que comeca com digito (ex.: '2contador') eh erro LEXICO e ja
   foi reportado pelo Flex, que mesmo assim o devolve como T_ID para a analise
   sintatica continuar. O modulo semantico o ignora para nao cadastra-lo na
   Tabela de Simbolos nem gerar erros semanticos em cascata.
   -------------------------------------------------------------------------- */
static int identificador_invalido(const char *nome) {
    return isdigit((unsigned char)nome[0]);
}

/* --------------------------------------------------------------------------
   1. BUSCA NA TABELA DE SIMBOLOS
   Percorre a tabela procurando pelo nome exato da variavel.
   Retorna a posicao no vetor se encontrar, ou -1 se nao existir.
   -------------------------------------------------------------------------- */
int buscar_simbolo(const char *nome) {
    for (int i = 0; i < n_simbolos; i++) {
        if (strcmp(tabela[i].nome, nome) == 0) {
            return i;
        }
    }
    return -1;
}

/* --------------------------------------------------------------------------
   2. DECLARACAO DE VARIAVEL (Ex.: int a, b;)
   Regra semantica verificada:
   - Uma variavel nao pode ser declarada mais de uma vez.
   -------------------------------------------------------------------------- */
void declarar_variavel(const char *nome, TipoDado tipo, int linha) {
    /* Identificador malformado (ex.: '2contador') ja foi reportado como erro
       lexico: nao entra na Tabela de Simbolos. */
    if (identificador_invalido(nome)) {
        return;
    }

    int idx = buscar_simbolo(nome);
    if (idx != -1) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "variavel '%s' ja foi declarada anteriormente na linha %d",
                 nome, tabela[idx].linha_decl);
        erro_semantico(linha, msg);
        return;
    }

    if (n_simbolos >= MAX_SIMBOLOS) {
        erro_semantico(linha, "limite maximo da Tabela de Simbolos atingido");
        return;
    }

    tabela[n_simbolos].nome = strdup(nome);
    tabela[n_simbolos].tipo = tipo;
    tabela[n_simbolos].linha_decl = linha;
    n_simbolos++;
}

/* --------------------------------------------------------------------------
   3. USO DE VARIAVEL EM EXPRESSAO (Ex.: x + 10)
   Regra semantica verificada:
   - A variavel precisa ter sido declarada antes do uso.
   Retorna o TipoDado da variavel (ou TIPO_ERRO se nao declarada).
   -------------------------------------------------------------------------- */
TipoDado obter_tipo_variavel(const char *nome, int linha) {
    /* Erro lexico ja reportado: propaga TIPO_ERRO sem mensagem repetida. */
    if (identificador_invalido(nome)) {
        return TIPO_ERRO;
    }

    int idx = buscar_simbolo(nome);
    if (idx == -1) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "variavel '%s' usada sem ter sido declarada", nome);
        erro_semantico(linha, msg);
        return TIPO_ERRO;
    }
    return tabela[idx].tipo;
}

/* --------------------------------------------------------------------------
   4. COMPATIBILIDADE DE TIPOS EM OPERACOES ARITMETICAS (+, -, *, /)
   Regra semantica verificada (Tipagem Estrita):
   - Ambos os lados devem ter o mesmo tipo (int com int, ou float com float).
   - Se algum lado ja for TIPO_ERRO, apenas propaga TIPO_ERRO sem emitir
     nova mensagem repetida.
   -------------------------------------------------------------------------- */
TipoDado verificar_operacao_aritmetica(TipoDado esq, TipoDado dir, const char *op, int linha) {
    if (esq == TIPO_ERRO || dir == TIPO_ERRO) {
        return TIPO_ERRO;
    }

    if (esq != dir) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "operacao '%s' incompativel entre '%s' e '%s' (tipagem estrita)",
                 op, nome_tipo(esq), nome_tipo(dir));
        erro_semantico(linha, msg);
        return TIPO_ERRO;
    }

    return esq; /* Como esq == dir, o resultado mantem o mesmo tipo. */
}

/* --------------------------------------------------------------------------
   5. COMPATIBILIDADE DE TIPOS EM ATRIBUICAO (Ex.: limite = 10 + 5;)
   Regras semanticas verificadas:
   - A variavel que recebe o valor (lado esquerdo) deve estar declarada.
   - O tipo da variavel deve ser identico ao tipo resultante da expressao
     do lado direito (tipagem estrita).
   -------------------------------------------------------------------------- */
void verificar_atribuicao(const char *nome_var, TipoDado tipo_expr, int linha) {
    /* Erro lexico ja reportado: evita erro semantico em cascata. */
    if (identificador_invalido(nome_var)) {
        return;
    }

    int idx = buscar_simbolo(nome_var);
    if (idx == -1) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "variavel '%s' recebeu atribuicao sem ter sido declarada",
                 nome_var);
        erro_semantico(linha, msg);
        return;
    }

    /* Se a expressao da direita ja teve erro de tipo, nao reportamos segundo erro. */
    if (tipo_expr == TIPO_ERRO) {
        return;
    }

    TipoDado tipo_var = tabela[idx].tipo;
    if (tipo_var != tipo_expr) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "atribuicao incompativel para '%s': variavel eh '%s', mas expressao eh '%s'",
                 nome_var, nome_tipo(tipo_var), nome_tipo(tipo_expr));
        erro_semantico(linha, msg);
    }
}

/* --------------------------------------------------------------------------
   6. COMPATIBILIDADE DE TIPOS EM CONDICOES RELACIONAIS (==, !=, >, <, >=, <=)
   Regra semantica verificada (Tipagem Estrita):
   - So permite comparar grandezas do mesmo tipo (int com int, float com float).
   -------------------------------------------------------------------------- */
void verificar_condicao_relacional(TipoDado esq, TipoDado dir, const char *op, int linha) {
    if (esq == TIPO_ERRO || dir == TIPO_ERRO) {
        return;
    }

    if (esq != dir) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "condicao '%s' incompativel entre '%s' e '%s' (tipagem estrita)",
                 op, nome_tipo(esq), nome_tipo(dir));
        erro_semantico(linha, msg);
    }
}

/* --------------------------------------------------------------------------
   7. VERIFICACAO DO COMANDO DE SELECAO MULTIPLA (switch)
   Regra semantica verificada:
   - Como os casos (case 1:, case 2:) sao constantes inteiras, a expressao
     avaliada no 'switch (expr)' deve ser obrigatoriamente do tipo 'int'.
   -------------------------------------------------------------------------- */
void verificar_selecao_switch(TipoDado tipo_expr, int linha) {
    if (tipo_expr == TIPO_ERRO) {
        return;
    }

    if (tipo_expr != TIPO_INT) {
        char msg[160];
        snprintf(msg, sizeof(msg),
                 "expressao do 'switch' deve ser 'int', mas recebeu '%s'",
                 nome_tipo(tipo_expr));
        erro_semantico(linha, msg);
    }
}

/* --------------------------------------------------------------------------
   8. TRADUCAO DIRIGIDA POR SINTAXE: NOTACAO POS-FIXADA (RPN)
   Na notacao pos-fixada (Polonesa Reversa), os operandos vem primeiro e o
   operador vem DEPOIS:
     Infixa     : 10 + 5 * 2
     Pos-Fixada : 10 5 2 * +
   Por que ela eh excelente para compiladores?
   Porque dispensa parenteses: a ordem em que o Bison reduz as regras (guiada
   pela precedencia de '*' sobre '+' e pelos parenteses) ja monta a string
   pos-fixada na ordem exata de execucao!
   -------------------------------------------------------------------------- */
char* combinar_rpn(char *esq, char *dir, const char *op) {
    size_t tam = strlen(esq) + strlen(dir) + strlen(op) + 3;
    char *nova = (char *)malloc(tam);
    if (nova) {
        snprintf(nova, tam, "%s %s %s", esq, dir, op);
    }
    free(esq);
    free(dir);
    return nova;
}

void registrar_traducao(int linha, const char *contexto, const char *rpn) {
    if (n_traducoes >= MAX_TRADUCOES) {
        return;
    }
    size_t tam = strlen(contexto) + strlen(rpn) + 16;
    char *buf = (char *)malloc(tam);
    if (buf) {
        snprintf(buf, tam, "%-14s %s", contexto, rpn);
        traducoes[n_traducoes].linha = linha;
        traducoes[n_traducoes].texto = buf;
        n_traducoes++;
    }
}

void imprimir_traducao_posfixada(void) {
    printf("\n===== TRADUCAO DIRIGIDA POR SINTAXE (POS-FIXADA) =====\n");
    if (n_traducoes == 0) {
        printf("(nenhuma expressao traduzida)\n");
        return;
    }
    printf("%-8s %-14s %s\n", "LINHA", "CONSTRUCAO", "NOTACAO POS-FIXADA (RPN)");
    for (int i = 0; i < n_traducoes; i++) {
        printf("%-8d %s\n", traducoes[i].linha, traducoes[i].texto);
    }
}

/* --------------------------------------------------------------------------
   IMPRESSAO E LIMPEZA DE MEMORIA
   -------------------------------------------------------------------------- */
void imprimir_tabela_simbolos(void) {
    printf("\n===== TABELA DE SIMBOLOS =====\n");
    if (n_simbolos == 0) {
        printf("(vazia)\n");
        return;
    }
    printf("%-20s %-10s %s\n", "NOME", "TIPO", "LINHA DECL.");
    for (int i = 0; i < n_simbolos; i++) {
        printf("%-20s %-10s %d\n",
               tabela[i].nome,
               nome_tipo(tabela[i].tipo),
               tabela[i].linha_decl);
    }
}

void liberar_memoria_semantica(void) {
    for (int i = 0; i < n_simbolos; i++) {
        free(tabela[i].nome);
        tabela[i].nome = NULL;
    }
    n_simbolos = 0;

    for (int i = 0; i < n_traducoes; i++) {
        free(traducoes[i].texto);
        traducoes[i].texto = NULL;
    }
    n_traducoes = 0;
}


## 3. Analisador Léxico (`analisador.l` — Flex)
Reconhece as palavras reservadas da linguagem **`bell`** (`int`, `float`, `if`, `else`, `switch`, `case`, `break`, `default`), operadores aritméticos, os 6 operadores relacionais, literais numéricos e identificadores.


In [ ]:
%%writefile analisador.l
%{
/* ==========================================================================
   ANALISADOR LEXICO (FLEX) -- Linguagem bell
   Trabalho do Grau B -- Compiladores -- UNISINOS

   Papel deste arquivo (analisador.l) no Grau B:
   1) Agrupa os caracteres do arquivo em lexemas;
   2) Rastreia linha (yylineno) e colunas (token_col_ini, token_col_fim);
   3) Identifica a categoria (token) de cada lexema;
   4) Para identificadores e literais numericos, guarda o texto em 'yylval.texto'
      (usado na Tabela de Simbolos e na Traducao Pos-Fixada);
   5) Devolve o codigo do token (ex.: T_INT, T_ID, T_MAIS) para o Bison!
   ========================================================================== */

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "semantico.h"
#include "analisador.tab.h" /* Cabecalho gerado pelo Bison com os tokens T_... */

int erros_lexicos = 0;      /* Contador de erros lexicos */
int num_tokens = 0;         /* Contador de tokens validos reconhecidos */

/* Controle de posicao de colunas (base 1) e modo de listagem de tokens (--tokens) */
int listar_tokens = 0;
int coluna_atual  = 1;
int token_col_ini = 1;
int token_col_fim = 1;

/* --------------------------------------------------------------------------
   RASTREIO DOS DOIS ULTIMOS TOKENS ENTREGUES AO BISON
   Por que isso existe? Quando o Bison detecta um erro, o Flex ja pode ter
   avancado alem do ponto do erro (ex.: consumiu a quebra de linha final e
   chegou ao fim do arquivo, ou ja leu o primeiro token da linha seguinte
   quando falta o ';'). Nesses casos 'yylineno' aponta para a linha errada.
   Guardando a linha de cada token devolvido, o yyerror() consegue informar
   a linha onde o problema realmente esta.
   - tok_atual / linha_tok_atual        : ultimo token devolvido (lookahead);
   - tok_anterior / linha_tok_anterior  : token devolvido antes dele.
   -------------------------------------------------------------------------- */
int tok_atual = 0, linha_tok_atual = 1;
int tok_anterior = 0, linha_tok_anterior = 1;

#define RETORNA(t) do { \
    tok_anterior = tok_atual; \
    linha_tok_anterior = linha_tok_atual; \
    tok_atual = (t); \
    linha_tok_atual = yylineno; \
    return (t); \
} while (0)

/* YY_USER_ACTION eh executada automaticamente pelo Flex antes de cada regra,
   atualizando a coluna inicial e final do lexema recem-lido. */
#define YY_USER_ACTION \
    token_col_ini = coluna_atual; \
    for (yy_size_t i = 0; i < (yy_size_t)yyleng; i++) { \
        if (yytext[i] == '\n') { \
            coluna_atual = 1; \
        } else { \
            coluna_atual++; \
        } \
    } \
    token_col_fim = coluna_atual - 1;

void registrar_token_lexico(const char *categoria) {
    if (listar_tokens) {
        printf("%d %d %d %-20s %s\n",
               yylineno, token_col_ini, token_col_fim, categoria, yytext);
    }
}

void erro_lexico(const char *msg) {
    if (!listar_tokens) {
        printf("  [%s - ERRO LEXICO] linha %d: %s\n", NOME_LINGUAGEM, yylineno, msg);
    }
    erros_lexicos++;
}
%}

%option noyywrap yylineno
%option nounput noinput

/* Apelidos para expressoes regulares basicas */
LETRA   [A-Za-z_]
DIGITO  [0-9]
ID      {LETRA}({LETRA}|{DIGITO})*

%%

 /* -------------------------------------------------------------------------
    1. PALAVRAS RESERVADAS DA LINGUAGEM (Estilo C)
    Devem vir ANTES da regra de identificador ({ID}) para terem prioridade.
    ------------------------------------------------------------------------- */
"int"       { num_tokens++; registrar_token_lexico("TIPO"); RETORNA(T_INT); }
"float"     { num_tokens++; registrar_token_lexico("TIPO"); RETORNA(T_FLOAT); }
"if"        { num_tokens++; registrar_token_lexico("PALAVRA_RESERVADA"); RETORNA(T_IF); }
"else"      { num_tokens++; registrar_token_lexico("PALAVRA_RESERVADA"); RETORNA(T_ELSE); }
"switch"    { num_tokens++; registrar_token_lexico("PALAVRA_RESERVADA"); RETORNA(T_SWITCH); }
"case"      { num_tokens++; registrar_token_lexico("PALAVRA_RESERVADA"); RETORNA(T_CASE); }
"break"     { num_tokens++; registrar_token_lexico("PALAVRA_RESERVADA"); RETORNA(T_BREAK); }
"default"   { num_tokens++; registrar_token_lexico("PALAVRA_RESERVADA"); RETORNA(T_DEFAULT); }

 /* -------------------------------------------------------------------------
    2. IDENTIFICADORES E ERRO LEXICO DE ID COMECANDO COM DIGITO
    ------------------------------------------------------------------------- */
{DIGITO}+{LETRA}({LETRA}|{DIGITO})*   {
    /* Exemplo: '2contador'. Erro lexico classico mantido do Grau A. */
    char m[128];
    snprintf(m, sizeof(m), "identificador nao pode comecar com digito: '%s'", yytext);
    erro_lexico(m);
    registrar_token_lexico("ERRO_LEXICO");
    yylval.texto = strdup(yytext);
    RETORNA(T_ID);
}

{ID}        {
    num_tokens++;
    registrar_token_lexico("IDENTIFICADOR");
    yylval.texto = strdup(yytext);
    RETORNA(T_ID);
}

 /* -------------------------------------------------------------------------
    3. LITERAIS NUMERICOS (FLOAT e INT)
    Guardamos tambem o texto do numero em yylval.texto para montar a
    Notacao Pos-Fixada (RPN) na traducao dirigida por sintaxe!
    ------------------------------------------------------------------------- */
{DIGITO}+"."{DIGITO}+   {
    num_tokens++;
    registrar_token_lexico("LITERAL_FLOAT");
    yylval.texto = strdup(yytext);
    RETORNA(T_NUM_FLOAT);
}

{DIGITO}+               {
    num_tokens++;
    registrar_token_lexico("LITERAL_INT");
    yylval.texto = strdup(yytext);
    RETORNA(T_NUM_INT);
}

 /* -------------------------------------------------------------------------
    4. OPERADORES RELACIONAIS (6 obrigatorios pelo enunciado)
    ------------------------------------------------------------------------- */
"=="        { num_tokens++; registrar_token_lexico("OP_RELACIONAL"); RETORNA(T_IGUAL); }
"!="        { num_tokens++; registrar_token_lexico("OP_RELACIONAL"); RETORNA(T_DIF); }
">="        { num_tokens++; registrar_token_lexico("OP_RELACIONAL"); RETORNA(T_MAIOR_IGUAL); }
"<="        { num_tokens++; registrar_token_lexico("OP_RELACIONAL"); RETORNA(T_MENOR_IGUAL); }
">"         { num_tokens++; registrar_token_lexico("OP_RELACIONAL"); RETORNA(T_MAIOR); }
"<"         { num_tokens++; registrar_token_lexico("OP_RELACIONAL"); RETORNA(T_MENOR); }

 /* -------------------------------------------------------------------------
    5. OPERADORES ARITMETICOS E DE ATRIBUICAO
    ------------------------------------------------------------------------- */
"="         { num_tokens++; registrar_token_lexico("OP_ATRIBUICAO"); RETORNA(T_ATRIB); }
"+"         { num_tokens++; registrar_token_lexico("OP_ARITMETICO"); RETORNA(T_MAIS); }
"-"         { num_tokens++; registrar_token_lexico("OP_ARITMETICO"); RETORNA(T_MENOS); }
"*"         { num_tokens++; registrar_token_lexico("OP_ARITMETICO"); RETORNA(T_MULT); }
"/"         { num_tokens++; registrar_token_lexico("OP_ARITMETICO"); RETORNA(T_DIV); }

 /* -------------------------------------------------------------------------
    6. DELIMITADORES E PONTUACAO
    ------------------------------------------------------------------------- */
"("         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_ABRE_PAR); }
")"         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_FECHA_PAR); }
"{"         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_ABRE_CHAVE); }
"}"         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_FECHA_CHAVE); }
":"         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_DOIS_PONTOS); }
","         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_VIRG); }
";"         { num_tokens++; registrar_token_lexico("DELIMITADOR"); RETORNA(T_PVIRG); }

 /* -------------------------------------------------------------------------
    7. ESPACOS EM BRANCO E COMENTARIOS (ignorados pelo sintatico)
    ------------------------------------------------------------------------- */
"//".*          { registrar_token_lexico("COMENTARIO"); }
[ \t\r\n]+      { /* Espacos, tabulacoes e quebras de linha: ignorados */ }

 /* -------------------------------------------------------------------------
    8. CARACTERE INVALIDO (Qualquer simbolo que nao pertence a linguagem)
    ------------------------------------------------------------------------- */
.           {
    char m[64];
    snprintf(m, sizeof(m), "caractere invalido '%s'", yytext);
    erro_lexico(m);
    registrar_token_lexico("ERRO_LEXICO");
}

%%


## 4. Analisador Sintático, Semântico e Tradutor (`analisador.y` — Bison)
Especifica a Gramática Livre de Contexto da linguagem **`bell`**, a precedência de operadores (`%left`), as chamadas às verificações semânticas e a geração da Notação Pós-Fixada.


In [ ]:
%%writefile analisador.y
%{
/* ==========================================================================
   ANALISADOR SINTATICO, SEMANTICO E TRADUTOR (BISON) -- Linguagem bell
   Trabalho do Grau B -- Compiladores -- UNISINOS

   Construcoes reconhecidas neste arquivo:
   a) Declaracao de variaveis (int, float) individuais ou em lista;
   b) Comando condicional (if com bloco else opcional);
   c) Comando de selecao multipla (switch / case / break / default opcional);
   d) Expressoes aritmeticas (+, -, *, /) com precedencia e parenteses;
   e) Atribuicao a variaveis declaradas (=);
   f) Condicoes com os 6 operadores relacionais (==, !=, >, <, >=, <=).

   Traducao Dirigida por Sintaxe:
   - Gera e registra a Notacao Pos-Fixada (Polonesa Reversa / RPN) de todas
     as expressoes, atribuicoes, condicoes e selecoes do programa!
   ========================================================================== */

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "semantico.h"

/* Declaracoes vindas do Flex (analisador.l) */
extern int yylex(void);
extern int yylineno;
extern char *yytext;
extern FILE *yyin;
extern int erros_lexicos;
extern int num_tokens;
extern int listar_tokens;
extern int tok_atual, linha_tok_atual;       /* Rastreio de tokens (analisador.l) */
extern int tok_anterior, linha_tok_anterior;

/* Contador de erros sintaticos (estruturais) */
int erros_sintaticos = 0;

/* Guarda temporariamente o tipo da declaracao em curso (ex.: em 'int a, b;',
   guarda TIPO_INT para aplicar tanto em 'a' quanto em 'b'). */
static TipoDado tipo_declaracao_atual;

/* Funcao chamada automaticamente pelo Bison quando encontra erro sintatico. */
void yyerror(const char *msg);
%}

/* --------------------------------------------------------------------------
   %union: Define a "mochila" (yylval) que carrega valores do Flex para o Bison
   e entre as regras da arvore sintatica.
   - 'texto' : strings de identificadores, numeros e operadores relacionais;
   - 'tipo'  : TipoDado (TIPO_INT, TIPO_FLOAT, TIPO_ERRO);
   - 'expr'  : AtributoExpressao (carrega juntos o 'tipo' semantico e a
               string 'rpn' da Traducao Pos-Fixada!).
   -------------------------------------------------------------------------- */
%union {
    char *texto;
    TipoDado tipo;
    AtributoExpressao expr;
}

/* --------------------------------------------------------------------------
   DECLARACAO DOS TOKENS (Simbolos Terminais vindos do Flex)
   -------------------------------------------------------------------------- */
%token T_INT T_FLOAT
%token T_IF T_ELSE
%token T_SWITCH T_CASE T_BREAK T_DEFAULT
%token <texto> T_ID T_NUM_INT T_NUM_FLOAT
%token T_IGUAL T_DIF T_MAIOR T_MENOR T_MAIOR_IGUAL T_MENOR_IGUAL
%token T_ATRIB
%token T_MAIS T_MENOS T_MULT T_DIV
%token T_ABRE_PAR T_FECHA_PAR T_ABRE_CHAVE T_FECHA_CHAVE
%token T_DOIS_PONTOS T_VIRG T_PVIRG

/* --------------------------------------------------------------------------
   TIPAGEM DOS NAO-TERMINAIS (Simbolos da Gramatica)
   -------------------------------------------------------------------------- */
%type <tipo> tipo
%type <expr> expressao
%type <texto> op_relacional

/* --------------------------------------------------------------------------
   PRECEDENCIA E ASSOCIATIVIDADE DE OPERADORES
   1) '+' e '-' tem menor precedencia e sao associativos a esquerda (%left).
   2) '*' e '/' tem MAIOR precedencia que '+' e '-' e tambem sao %left.
   -------------------------------------------------------------------------- */
%left T_MAIS T_MENOS
%left T_MULT T_DIV

%%

/* ==========================================================================
   REGRAS DA GRAMATICA + ACOES SEMANTICAS + TRADUCAO DIRIGIDA POR SINTAXE
   ========================================================================== */

programa
    : lista_comandos
    ;

lista_comandos
    : /* vazio: permite bloco vazio ou fim da recursao */
    | lista_comandos comando
    ;

comando
    : declaracao
    | atribuicao
    | condicional
    | selecao_multipla
    | error T_PVIRG {
        /* Recuperacao de erro sintatico ate o proximo ';' */
        yyerrok;
    }
    ;

/* --------------------------------------------------------------------------
   1. DECLARACAO DE VARIAVEIS (Ex.: int limite, faixa; ou float media;)
   -------------------------------------------------------------------------- */
declaracao
    : tipo lista_ids T_PVIRG
    ;

tipo
    : T_INT {
        $$ = TIPO_INT;
        tipo_declaracao_atual = TIPO_INT;
    }
    | T_FLOAT {
        $$ = TIPO_FLOAT;
        tipo_declaracao_atual = TIPO_FLOAT;
    }
    ;

lista_ids
    : T_ID {
        declarar_variavel($1, tipo_declaracao_atual, yylineno);
        free($1);
    }
    | lista_ids T_VIRG T_ID {
        declarar_variavel($3, tipo_declaracao_atual, yylineno);
        free($3);
    }
    ;

/* --------------------------------------------------------------------------
   2. ATRIBUICAO (Ex.: limite = 10 + 5 * 2;)
   - Verificacao Semantica: confere se $1 foi declarada e tem o mesmo tipo de $3.tipo.
   - Traducao Pos-Fixada: monta "<id> <expr_rpn> =" (ex.: "limite 10 5 2 * + =").
   -------------------------------------------------------------------------- */
atribuicao
    : T_ID T_ATRIB expressao T_PVIRG {
        verificar_atribuicao($1, $3.tipo, yylineno);

        char *rpn_atrib = combinar_rpn($1, $3.rpn, "=");
        registrar_traducao(yylineno, "[ATRIBUICAO]", rpn_atrib);
        free(rpn_atrib);
    }
    ;

/* --------------------------------------------------------------------------
   3. COMANDO CONDICIONAL (if com bloco else opcional)
   -------------------------------------------------------------------------- */
condicional
    : T_IF T_ABRE_PAR condicao T_FECHA_PAR bloco bloco_else_opc
    ;

bloco
    : T_ABRE_CHAVE lista_comandos T_FECHA_CHAVE
    ;

bloco_else_opc
    : /* vazio: quando o 'if' nao tem 'else' */
    | T_ELSE bloco
    ;

/* --------------------------------------------------------------------------
   4. CONDICOES COM OS 6 OPERADORES RELACIONAIS (==, !=, >, <, >=, <=)
   - Verificacao Semantica: compara $1.tipo com $3.tipo.
   - Traducao Pos-Fixada: monta "<esq_rpn> <dir_rpn> <op>" (ex.: "limite 20 >").
   -------------------------------------------------------------------------- */
condicao
    : expressao op_relacional expressao {
        verificar_condicao_relacional($1.tipo, $3.tipo, $2, yylineno);

        char *rpn_cond = combinar_rpn($1.rpn, $3.rpn, $2);
        registrar_traducao(yylineno, "[CONDICAO IF]", rpn_cond);
        free(rpn_cond);
    }
    ;

op_relacional
    : T_IGUAL       { $$ = "=="; }
    | T_DIF         { $$ = "!="; }
    | T_MAIOR       { $$ = ">";  }
    | T_MENOR       { $$ = "<";  }
    | T_MAIOR_IGUAL { $$ = ">="; }
    | T_MENOR_IGUAL { $$ = "<="; }
    ;

/* --------------------------------------------------------------------------
   5. COMANDO DE SELECAO MULTIPLA (switch / case / break / default)
   -------------------------------------------------------------------------- */
selecao_multipla
    : cabecalho_switch T_ABRE_CHAVE lista_casos caso_padrao_opc T_FECHA_CHAVE
    ;

cabecalho_switch
    : T_SWITCH T_ABRE_PAR expressao T_FECHA_PAR {
        verificar_selecao_switch($3.tipo, yylineno);
        registrar_traducao(yylineno, "[SWITCH]", $3.rpn);
        free($3.rpn);
    }
    ;

lista_casos
    : caso
    | lista_casos caso
    ;

caso
    : rotulos_caso lista_comandos T_BREAK T_PVIRG
    ;

rotulos_caso
    : T_CASE lista_inteiros T_DOIS_PONTOS
    | rotulos_caso T_CASE lista_inteiros T_DOIS_PONTOS
    ;

lista_inteiros
    : T_NUM_INT {
        free($1);
    }
    | lista_inteiros T_VIRG T_NUM_INT {
        free($3);
    }
    ;

caso_padrao_opc
    : /* vazio: default eh opcional */
    | T_DEFAULT T_DOIS_PONTOS lista_comandos break_opc
    ;

break_opc
    : /* vazio: permite default sem break */
    | T_BREAK T_PVIRG
    ;

/* --------------------------------------------------------------------------
   6. EXPRESSOES ARITMETICAS (+, -, *, /, parenteses, literais e variaveis)
   Repare na elegancia da Traducao Dirigida por Sintaxe:
   Cada regra calcula:
     1) $$.tipo -> Validacao semantica dos tipos;
     2) $$.rpn  -> String na Notacao Pos-Fixada (Polonesa Reversa).
   -------------------------------------------------------------------------- */
expressao
    : expressao T_MAIS expressao {
        $$.tipo = verificar_operacao_aritmetica($1.tipo, $3.tipo, "+", yylineno);
        $$.rpn  = combinar_rpn($1.rpn, $3.rpn, "+");
    }
    | expressao T_MENOS expressao {
        $$.tipo = verificar_operacao_aritmetica($1.tipo, $3.tipo, "-", yylineno);
        $$.rpn  = combinar_rpn($1.rpn, $3.rpn, "-");
    }
    | expressao T_MULT expressao {
        $$.tipo = verificar_operacao_aritmetica($1.tipo, $3.tipo, "*", yylineno);
        $$.rpn  = combinar_rpn($1.rpn, $3.rpn, "*");
    }
    | expressao T_DIV expressao {
        $$.tipo = verificar_operacao_aritmetica($1.tipo, $3.tipo, "/", yylineno);
        $$.rpn  = combinar_rpn($1.rpn, $3.rpn, "/");
    }
    | T_ABRE_PAR expressao T_FECHA_PAR {
        /* Na notacao pos-fixada, parenteses nao precisam aparecer no texto,
           pois eles ja cumpriram seu papel de alterar a ordem de reducao! */
        $$ = $2;
    }
    | T_NUM_INT {
        $$.tipo = TIPO_INT;
        $$.rpn  = $1;
    }
    | T_NUM_FLOAT {
        $$.tipo = TIPO_FLOAT;
        $$.rpn  = $1;
    }
    | T_ID {
        $$.tipo = obter_tipo_variavel($1, yylineno);
        $$.rpn  = $1;
    }
    ;

%%

/* --------------------------------------------------------------------------
   TRATAMENTO DE ERRO SINTATICO
   -------------------------------------------------------------------------- */
/* Tokens que podem encerrar uma expressao ou declaracao (ex.: 'b = 20', 'int a').
   Se um deles for seguido por algo inesperado, o mais provavel eh que falte ';'. */
static int encerra_expressao(int tok) {
    return tok == T_ID || tok == T_NUM_INT || tok == T_NUM_FLOAT;
}

void yyerror(const char *msg) {
    (void)msg;
    erros_sintaticos++;

    /* Caso 1: erro detectado no FIM DO ARQUIVO.
       O Flex ja consumiu as quebras de linha finais, entao yylineno aponta
       para depois da ultima linha. Usamos a linha do ultimo token lido. */
    if (yychar == YYEOF) {
        if (encerra_expressao(tok_atual)) {
            printf("  [%s - ERRO SINTATICO] linha %d: esperado ';' ao final do comando\n",
                   NOME_LINGUAGEM, linha_tok_atual);
        } else {
            printf("  [%s - ERRO SINTATICO] linha %d: fim de arquivo inesperado "
                   "(comando ou bloco nao encerrado)\n",
                   NOME_LINGUAGEM, linha_tok_atual);
        }
        return;
    }

    /* Caso 2: falta de ';' antes de uma quebra de linha.
       Ex.: 'b = 20' (linha 5) seguido de 'a = b + 1;' (linha 6). O erro so eh
       percebido ao ler 'a' na linha 6, mas o problema esta no fim da linha 5. */
    if (linha_tok_atual > linha_tok_anterior && encerra_expressao(tok_anterior)) {
        printf("  [%s - ERRO SINTATICO] linha %d: esperado ';' ao final do comando "
               "(encontrado '%s' na linha %d)\n",
               NOME_LINGUAGEM, linha_tok_anterior, yytext, linha_tok_atual);
        return;
    }

    /* Caso geral: o token inesperado esta na propria linha do erro. */
    printf("  [%s - ERRO SINTATICO] linha %d: token inesperado '%s'\n",
           NOME_LINGUAGEM, linha_tok_atual, yytext);
}

/* --------------------------------------------------------------------------
   FUNCAO PRINCIPAL
   -------------------------------------------------------------------------- */
int main(int argc, char **argv) {
    /* Modo de inspecao lexica (--tokens): imprime linha, colunas e categoria
       de cada token reconhecido pelo analisador lexico. */
    if (argc > 1 && strcmp(argv[1], "--tokens") == 0) {
        listar_tokens = 1;
        if (argc > 2) {
            yyin = fopen(argv[2], "r");
            if (!yyin) {
                perror("Erro ao abrir arquivo");
                return 1;
            }
        }
        int tok;
        while ((tok = yylex()) != 0) {
            if (tok == T_ID || tok == T_NUM_INT || tok == T_NUM_FLOAT) {
                free(yylval.texto);
            }
        }
        if (argc > 2 && yyin) {
            fclose(yyin);
        }
        return 0;
    }

    if (argc > 1) {
        yyin = fopen(argv[1], "r");
        if (!yyin) {
            perror("Erro ao abrir arquivo");
            return 1;
        }
        printf("Arquivo analisado (%s): %s\n", NOME_LINGUAGEM, argv[1]);
    }

    printf("\n===== ANALISE EM EXECUCAO =====\n");
    yyparse();

    imprimir_tabela_simbolos();

    int total_erros = erros_lexicos + erros_sintaticos + erros_semanticos;

    /* Se o programa eh valido, exibe a Traducao Dirigida por Sintaxe completa;
       mesmo se houver erros, exibe o que foi traduzido para fins didaticos. */
    imprimir_traducao_posfixada();

    printf("\n===== RESUMO (%s) =====\n", NOME_LINGUAGEM);
    printf("Tokens reconhecidos : %d\n", num_tokens);
    printf("Erros lexicos       : %d\n", erros_lexicos);
    printf("Erros sintaticos    : %d\n", erros_sintaticos);
    printf("Erros semanticos    : %d\n\n", erros_semanticos);
    printf("%s\n", total_erros == 0 ? ">> PROGRAMA VALIDO" : ">> PROGRAMA INVALIDO");

    liberar_memoria_semantica();
    if (argc > 1 && yyin) {
        fclose(yyin);
    }
    return 0;
}


## 5. Compilação do Analisador (`Makefile`)


In [ ]:
%%writefile Makefile
# ==========================================================================
# Makefile -- Compilador bell (Flex + Bison + Semantico em C)
# Trabalho do Grau B -- Compiladores -- UNISINOS
# ==========================================================================

CC     = gcc
LEX    = flex
YACC   = bison

CFLAGS = -Wall -Wextra -g

TARGET   = analisador
FLEX_SRC = analisador.l
YACC_SRC = analisador.y
SEM_SRC  = semantico.c

LEX_C    = lex.yy.c
YACC_C   = analisador.tab.c
YACC_H   = analisador.tab.h

all: $(TARGET)

# Passo 3: Compila tudo junto (Bison + Flex + Modulo Semantico)
$(TARGET): $(YACC_C) $(LEX_C) $(SEM_SRC) semantico.h
	$(CC) $(CFLAGS) -o $(TARGET) $(YACC_C) $(LEX_C) $(SEM_SRC)

# Passo 1: Bison gera analisador.tab.c e analisador.tab.h (flag -d)
$(YACC_C) $(YACC_H): $(YACC_SRC) semantico.h
	$(YACC) -d $(YACC_SRC)

# Passo 2: Flex gera lex.yy.c (precisa de analisador.tab.h pronto)
$(LEX_C): $(FLEX_SRC) $(YACC_H) semantico.h
	$(LEX) $(FLEX_SRC)

clean:
	rm -f $(TARGET) $(LEX_C) $(YACC_C) $(YACC_H) *.o


In [ ]:
!make clean && make


## 6. Criação dos Programas de Teste (4 Válidos e 4 Inválidos)


In [ ]:
%%writefile examples/valido1.txt
// Teste Valido 1: Equivalente ao exemplo do enunciado (declaracoes, expressao, if/else e switch)
int limite, faixa;
float media;

limite = 10 + 5 * 2;
faixa = 1;

if (limite > 20) {
    media = 10.0 / 2.0;
} else {
    media = 0.0;
}

switch (faixa) {
    case 1:
        limite = limite + 1;
        break;
    default:
        limite = 0;
}


In [ ]:
%%writefile examples/valido2.txt
// Teste Valido 2: Precedencia de operadores e parenteses na Notacao Pos-Fixada (RPN)
int a, b, c;
float x, y, resultado;

a = 10 + 5 * 2;
b = (10 + 5) * 2;
c = (a - b) / 2 + 1;

x = 2.5;
y = 1.5;
resultado = (x + y) * 3.0 - 1.0 / 2.0;


In [ ]:
%%writefile examples/valido3.txt
// Teste Valido 3: Todos os 6 operadores relacionais (==, !=, >, <, >=, <=) e if sem/com else
int nota, frequencia, aprovado;
float media, minimo;

nota = 8;
frequencia = 85;
aprovado = 0;
media = 7.5;
minimo = 6.0;

if (nota == 10) {
    aprovado = 1;
}

if (nota != 0) {
    if (frequencia >= 75) {
        aprovado = 1;
    }
}

if (media < minimo) {
    aprovado = 0;
} else {
    if (media <= 10.0) {
        aprovado = 1;
    }
}


In [ ]:
%%writefile examples/valido4.txt
// Teste Valido 4: Comando switch com casos unitarios, multiplos e caso padrao
int opcao, pontos, bonus;

opcao = 2;
pontos = 10;
bonus = 0;

switch (opcao + 1) {
    case 1:
        bonus = 5;
        break;
    case 2, 3:
        bonus = 10;
        pontos = pontos + bonus;
        break;
    case 4:
    case 5:
        bonus = 20;
        break;
    default:
        bonus = 1;
        break;
}


In [ ]:
%%writefile examples/invalido1.txt
// Teste Invalido 1 (Semantico): Variavel usada sem declarar e redeclaracao de variavel
int contador, limite;
float taxa;
int contador;

limite = 10;
total = limite + 5;
taxa = desconto * 2.0;


In [ ]:
%%writefile examples/invalido2.txt
// Teste Invalido 2 (Semantico): Violacoes de tipagem estrita em expressoes, atribuicoes, if e switch
int inteiro;
float real;

inteiro = 10 + 2.5;
inteiro = 3.14;
real = 42;

if (inteiro >= 5.5) {
    inteiro = 1;
}

switch (real) {
    case 1:
        inteiro = 0;
        break;
}


In [ ]:
%%writefile examples/invalido3.txt
// Teste Invalido 3 (Sintatico): Erros estruturais na gramatica (falta de ';', expressao incompleta)
int a, b;

a = 10 + * 5;
b = 20
a = b + 1;


In [ ]:
%%writefile examples/invalido4.txt
// Teste Invalido 4 (Lexico + Semantico): Identificador com digito inicial e caractere invalido
int 2contador;
float valor;

valor = 10.5 @ 2.0;
valor = 10;


---
## 7. Demonstração em Execução (Sequência para o Vídeo)

### 7.1. Programas Válidos + Saída da Tradução Dirigida por Sintaxe (Pós-Fixada)


In [ ]:
print("=== CODIGO-FONTE: examples/valido1.txt ===")
!cat -n examples/valido1.txt
print("\n=== EXECUCAO DO ANALISADOR ===")
!./analisador examples/valido1.txt


In [ ]:
print("=== CODIGO-FONTE: examples/valido2.txt (Precedencia vs Parenteses na RPN) ===")
!cat -n examples/valido2.txt
print("\n=== EXECUCAO DO ANALISADOR ===")
!./analisador examples/valido2.txt


In [ ]:
!./analisador examples/valido3.txt
print("=" * 65)
!./analisador examples/valido4.txt


### 7.2. Programas Inválidos - Erros Semânticos (com mensagem e número da linha)


In [ ]:
print("=== CODIGO-FONTE: examples/invalido1.txt (Declaracao previa e Duplicidade) ===")
!cat -n examples/invalido1.txt
print("\n=== EXECUCAO DO ANALISADOR ===")
!./analisador examples/invalido1.txt


In [ ]:
print("=== CODIGO-FONTE: examples/invalido2.txt (Tipagem Estrita em Expressoes, Atribuicoes, IF e SWITCH) ===")
!cat -n examples/invalido2.txt
print("\n=== EXECUCAO DO ANALISADOR ===")
!./analisador examples/invalido2.txt


### 7.3. Programas Inválidos - Erros Sintáticos e Léxicos (com mensagem e número da linha)


In [ ]:
print("=== CODIGO-FONTE: examples/invalido3.txt (Erros Sintaticos) ===")
!cat -n examples/invalido3.txt
print("\n=== EXECUCAO DO ANALISADOR ===")
!./analisador examples/invalido3.txt


In [ ]:
print("=== CODIGO-FONTE: examples/invalido4.txt (Erros Lexicos + Sintatico + Semantico) ===")
!cat -n examples/invalido4.txt
print("\n=== EXECUCAO DO ANALISADOR ===")
!./analisador examples/invalido4.txt


### 7.4. Linha correta em erros detectados tardiamente (correção do feedback do Grau A)
Quando falta `;` no último comando, o Flex já consumiu a quebra de linha final ao chegar no fim do arquivo, e `yylineno` aponta para a linha seguinte. O analisador agora guarda a linha dos dois últimos tokens entregues ao Bison (macro `RETORNA` em `analisador.l`) e o `yyerror` informa a linha onde o comando ficou incompleto.

Além disso, identificadores com erro léxico (ex.: `2x`) não entram mais na Tabela de Símbolos.

In [ ]:
# Arquivo de UMA linha, sem ';' e terminado por quebra de linha: o erro deve sair na linha 1 (e nao na 2)
!printf 'int a\n' > examples/fim_arquivo.txt
!cat -n examples/fim_arquivo.txt
!./analisador examples/fim_arquivo.txt | grep -E 'ERRO|>>'
print('=' * 65)
# Bloco nao fechado: o erro deve apontar a linha 3, ultima linha com codigo
!printf 'int a;\nif (a > 1) {\n    a = 2;\n\n' > examples/bloco_aberto.txt
!cat -n examples/bloco_aberto.txt
!./analisador examples/bloco_aberto.txt | grep -E 'ERRO|>>'
print('=' * 65)
# Identificador com erro lexico nao entra na Tabela de Simbolos
!printf 'int 2x, y;\ny = 1;\n' > examples/id_invalido.txt
!cat -n examples/id_invalido.txt
!./analisador examples/id_invalido.txt | grep -A4 -E 'ERRO|TABELA'